# Filter review: kept / discarded
Reads the `kept.csv` and `discarded.csv` written by `pixi run filter` (`eval_suite/filter_images.py`). Run all the cells: no widgets, the parameters are constants in the first cell.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from IPython.display import display

RESULTS = Path("results")               # --out folder of the scripts
OUT = RESULTS / "filtered"              # folder with kept.csv / discarded.csv
AGES = RESULTS / "e_age.csv"            # age estimates (optional, for the per-face details)
THUMB = (200, 120)                      # thumbnail (width, height)
COLS = 6                                # thumbnails per row
N = 60                                  # images shown per group (None = all)
SEED = 0                                # reproducible random sampling

kept = pd.read_csv(OUT / "kept.csv")
disc = pd.read_csv(OUT / "discarded.csv")
print(f"kept {len(kept)} | discarded {len(disc)} ({len(disc) / max(1, len(kept) + len(disc)):.0%})")
display(disc.reason.value_counts().to_frame("n"))

## Metric distributions
Used to choose the thresholds: `color_dev_frac_in` (kept vs discarded) and the probability of a minor (`p_minor`, only if the age table has that column).

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, col, title in [(axs[0], "color_dev_frac_in", "color_dev_frac_in"), (axs[1], "p_minor_max", "maximum p_minor per image")]:
    both = pd.concat([kept[col], disc[col]]).dropna() if col in kept else pd.Series(dtype=float)
    if both.empty: ax.set_visible(False); continue
    bins = np.linspace(0, max(both.max(), 1e-6), 30)
    ax.hist(kept[col].dropna(), bins=bins, alpha=.6, label="kept")
    ax.hist(disc[col].dropna(), bins=bins, alpha=.6, label="discarded")
    ax.set_yscale("log"); ax.set_title(title); ax.legend()
plt.tight_layout(); plt.show()
display(pd.concat([kept.assign(group="kept"), disc.assign(group="discarded")]).groupby("group")[["color_dev_frac_in"]].describe())

## Mosaics
Every thumbnail shows on top the file name, `color_dev_frac_in` and `p_minor`. The discarded ones are grouped by reason, randomly sampled.

In [ ]:
def thumb(r):
    try:
        im = Image.open(r.image_path).convert("RGB").resize(THUMB)
    except OSError:
        im = Image.new("RGB", THUMB, (60, 60, 60))
    d = ImageDraw.Draw(im)
    pm = getattr(r, "p_minor_max", float("nan"))
    txt = f"{Path(r.file).stem[:22]} cd={r.color_dev_frac_in:.2f} pm={pm:.2f}"
    d.rectangle([0, 0, THUMB[0], 13], fill=(0, 0, 0)); d.text((3, 1), txt, fill=(255, 255, 255))
    return im

def mosaic(df):
    df = df if N is None or len(df) <= N else df.sample(N, random_state=SEED)
    tiles = [thumb(r) for r in df.itertuples()]
    rows = -(-len(tiles) // COLS)
    canvas = Image.new("RGB", (COLS * THUMB[0], rows * THUMB[1]))
    for i, t in enumerate(tiles):
        canvas.paste(t, ((i % COLS) * THUMB[0], (i // COLS) * THUMB[1]))
    return canvas

for reason, g in disc.groupby("reason"):
    print(f"=== discarded: {reason} ({len(g)}) ===")
    display(mosaic(g))
print(f"=== kept ({len(kept)}) ===")
display(mosaic(kept))

## Inspecting a single frame
Set `FRAME` (part of the file name) to see all the rows of that frame, with the decision and the face details from `e_age.csv`.

In [ ]:
FRAME = "frame_000755"

allr = pd.concat([kept.assign(status="kept"), disc.assign(status="discarded")])
sel = allr[allr.file.str.contains(FRAME)]
display(sel[["file", "status", "reason", "n_est", "minor", "color_dev_frac_in"] + (["p_minor_max"] if "p_minor_max" in sel else [])])
if AGES.exists():
    res = pd.read_csv(AGES)
    res = res[res.file.map(lambda p: Path(p).name).isin(sel.file.map(lambda p: Path(p).name))]
    display(res[[c for c in ["file", "age_bin", "age_conf", "p_minor", "det_score"] if c in res]])
n = len(sel)
if n:
    fig, axs = plt.subplots(1, n, figsize=(4.5 * n, 3), squeeze=False)
    for ax, r in zip(axs[0], sel.itertuples()):
        ax.imshow(Image.open(r.image_path)); ax.axis("off")
        ax.set_title(f"{Path(r.file).stem[:28]} - {r.status}" + (f"\n{r.reason}" if isinstance(r.reason, str) else ""), fontsize=8)
    plt.tight_layout(); plt.show()